<a href="https://colab.research.google.com/github/talhanoor23/algorithmic-trading/blob/main/TradingBot_Using_macdEMA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Get_Realtime_Data**

In [ ]:
# 1. Install TA-Lib C library from conda-forge
!curl -L \
'https://anaconda.org/conda-forge/libta-lib/0.4.0/download/linux-64/libta-lib-0.4.0-h166bdaf_1.tar.bz2' \
| tar xj -C /usr/lib/x86_64-linux-gnu/ lib --strip-components=1

# 2. Install conda-package-handling and extract Python wrapper
!pip install conda-package-handling
!wget https://anaconda.org/conda-forge/ta-lib/0.5.1/download/linux-64/ta-lib-0.5.1-py311h9ecbd09_0.conda
!cph x ta-lib-0.5.1-py311h9ecbd09_0.conda

# 3. Move the extracted wrapper into site-packages
!mv ./ta-lib-0.5.1-py311h9ecbd09_0/lib/python3.11/site-packages/talib \
     /usr/local/lib/python3.11/dist-packages/

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  4107    0  4107    0     0  13113      0 --:--:-- --:--:-- --:--:-- 13121
100  517k  100  517k    0     0   539k      0 --:--:-- --:--:-- --:--:-- 1030k
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [conda-package-handling]
--2025-07-01 12:50:42--  https://anaconda.org/conda-forge/ta-lib/0.5.1/download/linux-64/ta-lib-0.5.1-py311h9ecbd09_0.conda
Resolving anaconda.org (anaconda.org)... 104.19.144.37, 104.19.145.37, 2606:4700::6813:9125, ...
Connecting to anaconda.org (anaconda.org)|104.19.144.37|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://binstar-cio-packages-prod.s3.amazonaws.com/5f738b4b78d4370a69f82984/673b62920c609ff6945bb24d?response-content-disposition=attachment%3B%20filename%3D%22ta-lib-0.5.1-py311h9ecbd09_0.conda%22%3B%20filename%2A%3DUTF-8%27%27ta-lib-0.5.1-py311h9ecbd09_

In [ ]:
import talib
print(len(talib.get_functions()), "functions available:")
print(talib.get_functions()[:5])   # shows first few indicators
print("\nGroups:", list(talib.get_function_groups().keys()))

158 functions available:
['HT_DCPERIOD', 'HT_DCPHASE', 'HT_PHASOR', 'HT_SINE', 'HT_TRENDMODE']

Groups: ['Cycle Indicators', 'Math Operators', 'Math Transform', 'Momentum Indicators', 'Overlap Studies', 'Pattern Recognition', 'Price Transform', 'Statistic Functions', 'Volatility Indicators', 'Volume Indicators']


In [ ]:
# !pip install --upgrade pip
# !pip install yfinance pandas_datareader
# !pip uninstall -y numpy pandas_ta
# !pip install numpy==1.24.3 pandas_ta

In [ ]:
from statsmodels.regression.rolling import RollingOLS
import pandas as pd
import pandas_ta as ta
import yfinance as yf
import numpy as np
import pandas_datareader.data as pdr
import matplotlib.pyplot as plt
import statsmodels.api as sm
import datetime as dt
import warnings
import websocket
warnings.filterwarnings('ignore')

In [ ]:
total_amount = 1000

longterm_al_amount = total_amount*0.80
longterm_cond = True
longterm_invested_amount = 0
longterm_portfolio = 0

shortterm_al_amount = total_amount*0.20
shortterm_inv_amount = 0
shortterm_portfolio = 0

total_amount_left = total_amount
main_portfolio = 0

investment = []
realtime_portfolio_value = []

In [ ]:
def buy(buying_amount, coin_price ):
    global total_amount_left, main_portfolio, investment
    quantity = buying_amount/coin_price
    total_amount_left -= buying_amount
    main_portfolio += quantity
    if investment == []:
        investment.append(buying_amount)
    else:
        investment.append(buying_amount)
        investment[-1] += investment[-2]


def sell(selling_amount, coin_price):
    global total_amount_left, main_portfolio, investment
    quantity = selling_amount/coin_price
    total_amount_left += selling_amount
    main_portfolio -= quantity
    investment.append(-selling_amount)
    investment[-1] += investment[-2]

In [ ]:
!pip install websocket-client

In [ ]:
import websocket
import json

closes = []
highs = []
lows = []

def on_message(ws, message):
    global closes, highs, lows, longterm_cond , longterm_al_amount, shortterm_al_amount, shortterm_portfolio, longterm_portfolio, total_amount_left, main_portfolio, investment, shortterm_inv_amount, longterm_invested_amount

    data = json.loads(message)

    # Check if the message contains kline data
    if "data" not in data or not isinstance(data["data"], list):
        return

    for kline in data["data"]:
        candle_closed = kline.get("confirm", False)

        if candle_closed:
            closes.append(float(kline["close"]))
            highs.append(float(kline["high"]))
            lows.append(float(kline["low"]))
            last_price = closes[-1]

            print("Candle closed!")
            print("Closes:", closes)
            print("Highs:", highs)
            print("Lows:", lows)
            print("Last Price:", last_price)

            if longterm_cond:
              buy(longterm_al_amount, closes[-1])
              longterm_portfolio += longterm_al_amount/closes[-1]
              longterm_invested_amount = longterm_portfolio * closes[-1]
              longterm_cond = False
              print("__ we bought $800 worth of bitcoin __")
            # else:
            #   sell(trade_amount, closes[-1])


 # Ensure enough candles are available
            if len(closes) > 30: #200
                np_closes = np.array(closes)

                # MACD Calculation
                macd, macd_signal, _ = talib.MACD(np_closes, fastperiod=12, slowperiod=26, signalperiod=9)
                last_macd = macd[-1]
                last_signal = macd_signal[-1]

                # EMA 200 Calculation
                ema200 = talib.EMA(np_closes, timeperiod=20) #200
                last_ema200 = ema200[-1]

                print(f"MACD: {last_macd}, Signal: {last_signal}, EMA200: {last_ema200}")



                # Buy Signal
                if last_macd > last_signal and last_price > last_ema200:
                    amt = shortterm_al_amount
                    remaining_short_amt = amt - shortterm_inv_amount
                    if remaining_short_amt > 0:
                        buy(remaining_short_amt, last_price)
                        shortterm_portfolio += remaining_short_amt/closes[-1]
                        print(f"__ we bought ${remaining_short_amt:.2f} worth of bitcoin __")

                # Sell Signal
                elif last_macd < last_signal and last_price < last_ema200:
                    amt = -shortterm_al_amount
                    remaining_short_amt = amt - shortterm_inv_amount
                    if remaining_short_amt < 0:
                        sell(remaining_short_amt, last_price)
                        shortterm_portfolio -= remaining_short_amt/closes[-1]
                        print(f"__ we sold ${remaining_short_amt:.2f} worth of bitcoin __")


                shortterm_inv_amount = (main_portfolio * last_price) - (longterm_portfolio * last_price)


                # Portfolio value update
                RT_portfolio_value = shortterm_inv_amount + (longterm_portfolio * last_price) + total_amount_left
                realtime_portfolio_value.append(RT_portfolio_value)

                print(f"Realtime Portfolio Value: {RT_portfolio_value:.2f}")
                print("--------------------------------------------------------------------------------")


def on_open(ws):
    # Subscribe to BTCUSDT 1-minute candlestick
    params = {
        "op": "subscribe",
        "args": ["kline.1.BTCUSDT"]
    }
    ws.send(json.dumps(params))

def on_close(ws, close_status_code, close_msg):
    print("WebSocket closed:", close_status_code, close_msg)

socket = "wss://stream.bybit.com/v5/public/linear"

ws = websocket.WebSocketApp(socket,
                            on_message=on_message,
                            on_open=on_open,
                            on_close=on_close)

In [ ]:
ws.run_forever()
#aroon timeperiod is set to 3, so it shows after 3 interval. greater the timepriod, prediction is more accurate or have high accuracy like in models, but here we deal with aroon indicators.

Candle closed!
Closes: [106743.5]
Highs: [106789.2]
Lows: [106734.6]
Last Price: 106743.5
__ we bought $800 worth of bitcoin __
Candle closed!
Closes: [106743.5, 106740.0]
Highs: [106789.2, 106743.5]
Lows: [106734.6, 106732.0]
Last Price: 106740.0
Candle closed!
Closes: [106743.5, 106740.0, 106760.7]
Highs: [106789.2, 106743.5, 106760.7]
Lows: [106734.6, 106732.0, 106725.8]
Last Price: 106760.7
Candle closed!
Closes: [106743.5, 106740.0, 106760.7, 106737.7]
Highs: [106789.2, 106743.5, 106760.7, 106765.1]
Lows: [106734.6, 106732.0, 106725.8, 106737.6]
Last Price: 106737.7
Candle closed!
Closes: [106743.5, 106740.0, 106760.7, 106737.7, 106710.4]
Highs: [106789.2, 106743.5, 106760.7, 106765.1, 106737.7]
Lows: [106734.6, 106732.0, 106725.8, 106737.6, 106710.0]
Last Price: 106710.4
Candle closed!
Closes: [106743.5, 106740.0, 106760.7, 106737.7, 106710.4, 106715.1]
Highs: [106789.2, 106743.5, 106760.7, 106765.1, 106737.7, 106745.3]
Lows: [106734.6, 106732.0, 106725.8, 106737.6, 106710.0, 106

True